In [ ]:
# 1. Install dependencies
!pip install -q --upgrade numpy tqdm scikit-learn

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm

# Setup Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# 2. MANUALLY SET DATASET PATH (The one you fixed!)
DATA_PATH = '/kaggle/input/datasets/sujeetkumar22/pestopia/Datasets/Pest_Dataset'

# 3. DINOv2 TRANSFORMS
# DINOv2 requires image sizes to be a multiple of 14. 224x224 is standard.
IMG_SIZE = 224 
BATCH_SIZE = 32 

def get_data():
    train_tfms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
        transforms.RandAugment(num_ops=2, magnitude=9), # State-of-the-art augmentation
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    full_ds = datasets.ImageFolder(DATA_PATH)
    train_sz = int(0.8 * len(full_ds))
    val_sz = len(full_ds) - train_sz
    
    generator = torch.Generator().manual_seed(42)
    train_idx, val_idx = random_split(range(len(full_ds)), [train_sz, val_sz], generator=generator)
    
    train_ds = datasets.ImageFolder(DATA_PATH, transform=train_tfms)
    val_ds = datasets.ImageFolder(DATA_PATH, transform=val_tfms)
    
    train_sub = torch.utils.data.Subset(train_ds, train_idx.indices)
    val_sub = torch.utils.data.Subset(val_ds, val_idx.indices)
    
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, full_ds.classes

train_loader, val_loader, class_names = get_data()
NUM_CLASSES = len(class_names)
print(f"✅ Data Loaded: {NUM_CLASSES} Classes")

# 4. BUILD CUSTOM DINOv2 MODEL
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Download Meta's DINOv2 Small model from PyTorch Hub
        print("Downloading Meta DINOv2 Weights...")
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        
        # DINOv2 ViT-Small outputs 384-dimensional features
        self.head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(384, num_classes)
        )

    def forward(self, x):
        # Extract features from the backbone
        features = self.backbone(x)
        # Classify the features
        return self.head(features)

model = DINOv2Classifier(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Training Loop Function
def run_epoch(loader, is_train, optimizer=None, scheduler=None):
    model.train() if is_train else model.eval()
    total_loss, correct = 0.0, 0
    iterator = tqdm(loader, desc="Training", leave=False) if is_train else loader
    
    with torch.set_grad_enabled(is_train):
        for img, label in iterator:
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            if is_train:
                optimizer.zero_grad()
                out = model(img)
                loss = criterion(out, label)
                loss.backward()
                optimizer.step()
                if scheduler is not None: scheduler.step()
            else:
                out = model(img)
                loss = criterion(out, label)
            
            total_loss += loss.item() * img.size(0)
            correct += (out.argmax(1) == label).sum().item()
            
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ====================================================
# STAGE 1: WARMUP (Train only the Head for 3 epochs)
# ====================================================
print("\n🔒 STAGE 1: Warmup...")
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer_warmup = optim.AdamW(model.head.parameters(), lr=1e-3)

for epoch in range(3):
    t_loss, t_acc = run_epoch(train_loader, True, optimizer_warmup)
    v_loss, v_acc = run_epoch(val_loader, False)
    print(f"Ep {epoch+1}/3 (Warmup) | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f}")

# ====================================================
# STAGE 2: FINE TUNING (Train entire model carefully)
# ====================================================
print("\n🔓 STAGE 2: Fine Tuning Backbone...")
for param in model.parameters():
    param.requires_grad = True

# Vision Transformers require very low learning rates for fine-tuning!
optimizer_ft = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=15 * len(train_loader), eta_min=1e-7)

best_acc = 0.0

for epoch in range(15):
    t_loss, t_acc = run_epoch(train_loader, True, optimizer_ft, scheduler_ft)
    v_loss, v_acc = run_epoch(val_loader, False)
    
    print(f"Ep {epoch+1}/15 (FineTune) | Val Acc: {v_acc:.4f} | Val Loss: {v_loss:.4f}")
    
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), 'dinov2_pest_model.pth')
        print(f"   🏆 Best Model Saved! ({best_acc*100:.2f}%)")

print(f"\n✅ Final Best Accuracy: {best_acc*100:.2f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 75.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 104.3 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 wh

/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 265MB/s]



🔒 STAGE 1: Warmup...


...

Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Ep 8/15 (FineTune) | Val Acc: 0.7625 | Val Loss: 1.6288
   🏆 Best Model Saved! (76.25%)


Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Ep 9/15 (FineTune) | Val Acc: 0.7646 | Val Loss: 1.6124
   🏆 Best Model Saved! (76.46%)


Training:   0%|          | 0/1398 [00:00<?, ?it/s]